In [5]:
from langchain.agents import create_agent
from langchain.agents.middleware import PIIMiddleware
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

from dotenv import load_dotenv

load_dotenv()

# 'block' - 检测到时抛出异常
# 'redact' - 替换为 [REDACTED_{PII_TYPE}]
# 'mask' - 部分屏蔽（例如，****-****-****-1234）
# 'hash' - 替换为确定性哈希
# apply_to_input 在模型调用之前检查用户消息
# apply_to_output 在模型调用之后检查 AI 消息


PII_email = PIIMiddleware(
  "email", 
  strategy="redact",
  apply_to_input=False, # 
  apply_to_output=True
)

PII_phone = PIIMiddleware(
  "phone_number",
  strategy="mask",
  # 自定义 detector 是对整段消息内容做 finditer，不能用 ^...$ 锚定，
  # 否则只有整条消息恰好等于手机号时才会命中；用 \d 前后环视限定边界即可
  detector=r"(?<!\d)1(?:3[0-9]|4[5-9]|5[0-35-9]|6[2567]|7[0-8]|8[0-9]|9[0-35-9])[0-9]{8}(?!\d)",
  apply_to_input=False,
  apply_to_output=True
)

agent = create_agent(
  model="deepseek-flash",
  checkpointer=InMemorySaver(),
  middleware=[PII_email, PII_phone]
)

response = agent.invoke(
  {"messages": [HumanMessage("我的邮箱是diken@qq.com，手机号是15912341234，你复述一遍")]},
  config={"configurable": {"thread_id": "thread_1"}}
)

for message in response["messages"]:
  message.pretty_print()

================================ Human Message =================================

我的邮箱是diken@qq.com，手机号是15912341234，你复述一遍
================================== Ai Message ==================================

你的邮箱是 [REDACTED_EMAIL]，手机号是 ****1234。
